In [33]:
import os
os.environ["HTTP_PROXY"]= "http://proxy.utwente.nl:3128"
os.environ["HTTPS_PROXY"]= "http://proxy.utwente.nl:3128"
os.environ["http_proxy"]= "http://proxy.utwente.nl:3128"
os.environ["https_proxy"]= "http://proxy.utwente.nl:3128"

In [34]:
import json
import requests

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
ATTACK_STIX_URL = "https://raw.githubusercontent.com/mitre/cti/master/enterprise-attack/enterprise-attack.json"
OUTPUT_FILE = "mitre_tactics_relationships.json"

# ------------------------------------------------------------
# 1) Download MITRE ATT&CK STIX bundle
# ------------------------------------------------------------
print("Downloading MITRE ATT&CK STIX bundle...")
resp = requests.get(ATTACK_STIX_URL, timeout=30)
resp.raise_for_status()

stix_data = resp.json()
objects = stix_data.get("objects", [])

# ------------------------------------------------------------
# 2) Extract tactics
# ------------------------------------------------------------
tactics = [
    obj for obj in objects
    if obj.get("type") == "x-mitre-tactic"
]

print(f"Found {len(tactics)} tactics")

# ------------------------------------------------------------
# 3) Convert to relationship format
# ------------------------------------------------------------
relationships = []

for tactic in tactics:
    tactic_id = tactic.get("external_references", [{}])[0].get("external_id")
    tactic_name = tactic.get("name")
    description = tactic.get("description", "").strip()
    shortname = tactic.get("x_mitre_shortname")

    if not tactic_id:
        continue

    relationships.append({
        "source_type": "mitre-attack",
        "source_name": "ATT&CK Tactic",
        "relationship_type": "defines",
        "relationship_description": description,
        "technique_id": tactic_id,                     # TAxxxx
        "technique_name": tactic_name,                 # e.g. "Initial Access"
        "technique_url": f"https://attack.mitre.org/tactics/{tactic_id}",
        "direction": "outgoing",
        "tactic_shortname": shortname                  # optional but useful
    })

# ------------------------------------------------------------
# 4) Save to disk
# ------------------------------------------------------------
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(relationships, f, indent=2, ensure_ascii=False)

print(f"Saved {len(relationships)} tactic relationships to {OUTPUT_FILE}")


Found 14 tactics
Saved 14 tactic relationships to mitre_tactics_relationships.json


Techniques

In [54]:
import json
import requests

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
ATTACK_STIX_URL = "https://raw.githubusercontent.com/mitre/cti/master/enterprise-attack/enterprise-attack.json"
OUTPUT_FILE = "mitre_techniques_relationships.json"

# ------------------------------------------------------------
# 1) Download MITRE ATT&CK STIX bundle
# ------------------------------------------------------------
print("Downloading MITRE ATT&CK STIX bundle...")
resp = requests.get(ATTACK_STIX_URL, timeout=30)
resp.raise_for_status()

stix_data = resp.json()
objects = stix_data.get("objects", [])

# ------------------------------------------------------------
# 2) Extract techniques (attack-pattern)
# ------------------------------------------------------------
techniques = [
    obj for obj in objects
    if obj.get("type") == "attack-pattern"
]

print(f"Found {len(techniques)} attack-pattern objects")

# ------------------------------------------------------------
# Helper: extract ATT&CK external ID
# ------------------------------------------------------------
def get_attack_id(obj):
    for ref in obj.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            return ref.get("external_id"), ref.get("url")
    return None, None

# ------------------------------------------------------------
# 3) Convert to relationship format
# ------------------------------------------------------------
relationships = []

for tech in techniques:
    tech_id, tech_url = get_attack_id(tech)
    if not tech_id:
        continue  # skip non-ATT&CK patterns
    
    # if subtech, then "technique_name" includes parent technique name
    if '.' in tech_id:
        parent_id = tech_id.split('.')[0]
        parent_tech = next((t for t in techniques if get_attack_id(t)[0] == parent_id), None)
        if parent_tech:
            parent_name = parent_tech.get("name")
            tech_name = tech.get("name")
            full_name = f"{parent_name} > {tech_name}"
        else:
            full_name = tech.get("name")
    relationships.append({
        "source_type": "attack-pattern",
        "relationship_description": tech.get("description", "").strip(),
        "technique_id": tech_id,                         # Txxxx or Txxxx.yyy
        "technique_name": full_name,
        "technique_url": tech_url or f"https://attack.mitre.org/techniques/{tech_id}",
        "direction": "incoming",
        "description_source": "technique_description"
        
    })

# ------------------------------------------------------------
# 4) Save to disk
# ------------------------------------------------------------
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(relationships, f, indent=2, ensure_ascii=False)

print(f"Saved {len(relationships)} technique relationships to {OUTPUT_FILE}")


Found 835 attack-pattern objects
Saved 835 technique relationships to mitre_techniques_relationships.json


Concat tec_with other data

In [60]:
with open("mitre_techniques_relationships.json", "r", encoding="utf-8") as f:
    tec_relationships = json.load(f)

with open("../../data_augmentatio_stefano/mitre/cleaned_capec.json", "r", encoding="utf-8") as f:
    old_relationships = json.load(f)

old_relationships = [
    item for item in old_relationships
    if item.get("direction") != "incoming"
]


for item in old_relationships:
    if item["direction"] == "incoming":
        print(item)

concatenated = tec_relationships + old_relationships

with open("../../data_augmentatio_stefano/mitre/mitre_relationships.json", "w", encoding="utf-8") as f:
    json.dump(concatenated, f, indent=2, ensure_ascii=False)